In [7]:
# ================= STEP 1: Mounting Google Drive - Giving Colab the key to warehouse =================

# Import Drive connector - lets Colab talk to your Google Drive
from google.colab import drive
# Import os - to check if paths exist without showing private folder names
import os

# Check: Starting mount
print("STEP 1 CHECK: Starting Google Drive mount...")

# Action: Mount Drive to /content/drive - will ask Allow, click Allow
drive.mount('/content/drive')

# Check: Confirm mount succeeded
print("\nSTEP 1 CHECK PASSED: Drive mounted at /content/drive")

# Privacy-safe check: Only True/False, no folder names shown
print(f"MyDrive path exists: {os.path.exists('/content/drive/MyDrive')} (True = success)")
print("STEP 1 COMPLETE: Ready for Step 2")

STEP 1 CHECK: Starting Google Drive mount...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

STEP 1 CHECK PASSED: Drive mounted at /content/drive
MyDrive path exists: True (True = success)
STEP 1 COMPLETE: Ready for Step 2


In [8]:
# ================= STEP 2: What tables exist inside regression.db =================
# This cell reads sqlite_master to list all drawers/tables without loading big data

# Import sqlite3 - tool to open.db filing cabinet
import sqlite3
# Import os - to check path exists
import os

# Set exact DB path - found in my Drive from my output
db_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/regression.db' # The exact path

# Check: Does DB file exist? True = good, False = Drive disconnected
print(f"STEP 2 CHECK: DB path exists? {os.path.exists(db_path)}")
print(f"STEP 2: Using DB at: {db_path}")

# Action: Connect to DB - opens cabinet, does not load data yet
conn = sqlite3.connect(db_path)
cursor = conn.cursor() # Cursor is like finger pointing inside cabinet

# Action: Query master list - sqlite_master holds names of all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

# Check: Print tables found
print("\nSTEP 2 CHECK: Tables found:")
for t in tables:
    print(f" - {t[0]}")

# Check: Is our Project 3 table there?
table_names = [t[0] for t in tables]
if 'predicting_sales' in table_names:
    print("\nSTEP 2 SUCCESS: 'predicting_sales' FOUND")
else:
    print("\nSTEP 2 FAILED: 'predicting_sales' NOT found")

# Action: Close connection
conn.close()
print("STEP 2 COMPLETE: Connection closed")

STEP 2 CHECK: DB path exists? True
STEP 2: Using DB at: /content/drive/MyDrive/INTTRVU/CAPSTONES/regression.db

STEP 2 CHECK: Tables found:
 - Insurance_Prediction
 - flight_fare
 - household_power_consumption
 - air_quality_index
 - predicting_sales
 - electric_motor_temperature
 - signal_metrics
 - mining_data
 - life_science
 - fuel_efficiency_automobile

STEP 2 SUCCESS: 'predicting_sales' FOUND
STEP 2 COMPLETE: Connection closed


In [9]:
# ================= STEP 3: Opening predicting_sales - peek at first 5 rows =================
# This cell shows total rows, column structure, and sample data for EDA start

# Import sqlite3 - to open DB
import sqlite3
# Import pandas - Excel-like tool in Python for tables
import pandas as pd
# Import os - to verify path
import os

# Set exact DB path
db_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/regression.db'

# Check: Path exists?
print(f"STEP 3 CHECK: DB exists? {os.path.exists(db_path)}")

# Action: Connect
conn = sqlite3.connect(db_path)

# Check 1: Total row count - COUNT(*) counts all rows
print("\nSTEP 3 CHECK 1: Total rows:")
count_df = pd.read_sql_query("SELECT COUNT(*) as total FROM predicting_sales", conn)
print(f"Total rows in predicting_sales: {count_df.iloc[0,0]}")

# Check 2: Column structure - PRAGMA shows column names and types
print("\nSTEP 3 CHECK 2: Column structure:")
struct_df = pd.read_sql_query("PRAGMA table_info(predicting_sales);", conn)
print(struct_df[['name','type']])

# Check 3: First 5 rows - LIMIT 5 is like preview in Excel
print("\nSTEP 3 CHECK 3: First 5 rows:")
sample_df = pd.read_sql_query("SELECT * FROM predicting_sales LIMIT 5", conn)
print(sample_df)

# Check 4: Quick stats for Sales - mean, min, max to understand sales range
print("\nSTEP 3 CHECK 4: Quick stats (first 100k rows for speed):")
stats_df = pd.read_sql_query("SELECT Sales, Customers FROM predicting_sales LIMIT 100000", conn)
print(stats_df.describe())

# Action: Close
conn.close()
print("\nSTEP 3 COMPLETE: Connection closed")

STEP 3 CHECK: DB exists? True

STEP 3 CHECK 1: Total rows:
Total rows in predicting_sales: 1017209

STEP 3 CHECK 2: Column structure:
            name     type
0          Store  INTEGER
1      DayOfWeek  INTEGER
2           Date     TEXT
3          Sales  INTEGER
4      Customers  INTEGER
5           Open  INTEGER
6          Promo  INTEGER
7   StateHoliday     TEXT
8  SchoolHoliday  INTEGER

STEP 3 CHECK 3: First 5 rows:
   Store  DayOfWeek        Date  Sales  Customers  Open  Promo StateHoliday  \
0      1          5  2015-07-31   5263        555     1      1            0   
1      2          5  2015-07-31   6064        625     1      1            0   
2      3          5  2015-07-31   8314        821     1      1            0   
3      4          5  2015-07-31  13995       1498     1      1            0   
4      5          5  2015-07-31   4822        559     1      1            0   

   SchoolHoliday  
0              1  
1              1  
2              1  
3              1  
4    

In [10]:
# ================= STEP 4: Data Cleaning - Checking for dirt in sales data =================
# This cell loads full 1M rows and checks nulls, closed stores, holidays, weird zeros

# Import tools - sqlite to open DB, pandas as Excel, os to check path
import sqlite3
import pandas as pd
import os

# Set exact DB path
db_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/regression.db'

# Check: Path exists?
print(f"STEP 4 CHECK: DB exists? {os.path.exists(db_path)}")

# Action: Connect and load full predicting_sales table into pandas DataFrame
# DataFrame = like a super Excel sheet in memory
conn = sqlite3.connect(db_path)
print("\nSTEP 4 ACTION: Loading 1,017,209 rows into pandas... (takes 20-30 sec)")
df = pd.read_sql_query("SELECT * FROM predicting_sales", conn)
conn.close()
print(f"Loaded! Shape = {df.shape} (rows, columns)")

# Check 1: Any NULL / missing values? isnull().sum() counts blanks per column
print("\nSTEP 4 CHECK 1: Missing values per column (0 is good):")
print(df.isnull().sum())

# Check 2: Open column - how many days stores were closed vs open?
print("\nSTEP 4 CHECK 2: Open vs Closed days:")
print(df['Open'].value_counts()) # value_counts counts 0s and 1s
print(f"Percent closed: { (df['Open']==0).mean()*100 :.2f}%")

# Check 3: When closed (Open=0), Sales and Customers should be 0. Is that true?
print("\nSTEP 4 CHECK 3: When Open=0, are Sales always 0?")
closed_df = df[df['Open']==0] # Filter only closed days
print(f"Closed rows: {len(closed_df)}")
print(f"Closed but Sales != 0 (should be 0): {(closed_df['Sales']!=0).sum()}")
print(f"Closed but Customers !=0 (should be 0): {(closed_df['Customers']!=0).sum()}")

# Check 4: When Open=1, any weird Sales=0? Shop open but no sale?
print("\nSTEP 4 CHECK 4: When Open=1, how many times Sales=0?")
open_df = df[df['Open']==1]
print(f"Open rows: {len(open_df)}")
print(f"Open but Sales=0: {(open_df['Sales']==0).sum()} (could be valid if no customer)")

# Check 5: Unique values in holiday and promo columns
print("\nSTEP 4 CHECK 5: Unique values (to understand categories):")
print(f"DayOfWeek unique: {sorted(df['DayOfWeek'].unique())}") # 1-7
print(f"Promo unique: {sorted(df['Promo'].unique())}") # 0/1
print(f"SchoolHoliday unique: {sorted(df['SchoolHoliday'].unique())}") # 0/1
print(f"StateHoliday unique: {sorted(df['StateHoliday'].unique())}") # Expect ['0','a','b','c']
print(f"Store count: {df['Store'].nunique()} unique stores")

# Check 6: Date range - min and max date to know years covered
print("\nSTEP 4 CHECK 6: Date range:")
# Convert Date TEXT to datetime for proper min/max - errors='coerce' makes bad dates NULL
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
print(f"Min Date: {df['Date'].min()}")
print(f"Max Date: {df['Date'].max()}")
print(f"Any bad dates that failed to parse? {df['Date'].isnull().sum()}")

# Check 7: Duplicates - same Store and Date twice?
print("\nSTEP 4 CHECK 7: Duplicate Store+Date combos?")
dups = df.duplicated(subset=['Store','Date']).sum()
print(f"Duplicate rows (same Store+Date): {dups}")

print("\nSTEP 4 COMPLETE: We now know what dirt exists")

STEP 4 CHECK: DB exists? True

STEP 4 ACTION: Loading 1,017,209 rows into pandas... (takes 20-30 sec)
Loaded! Shape = (1017209, 9) (rows, columns)

STEP 4 CHECK 1: Missing values per column (0 is good):
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

STEP 4 CHECK 2: Open vs Closed days:
Open
1    844392
0    172817
Name: count, dtype: int64
Percent closed: 16.99%

STEP 4 CHECK 3: When Open=0, are Sales always 0?
Closed rows: 172817
Closed but Sales != 0 (should be 0): 0
Closed but Customers !=0 (should be 0): 0

STEP 4 CHECK 4: When Open=1, how many times Sales=0?
Open rows: 844392
Open but Sales=0: 54 (could be valid if no customer)

STEP 4 CHECK 5: Unique values (to understand categories):
DayOfWeek unique: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Promo unique: [np.int64(0), np.int64(1)]
SchoolHolid

In [11]:
# ================= STEP 5: Feature Engineering - Making Date smart + Train/Eval split =================
# This cell creates new columns from Date and splits 700k / 200k / rest as per brief

# Import tools
import sqlite3
import pandas as pd
import os

# Set DB path
db_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/regression.db'

# Check: DB exists?
print(f"STEP 5 CHECK: DB exists? {os.path.exists(db_path)}")

# Action: Load data again (to get fresh copy after Step 4 conversion)
conn = sqlite3.connect(db_path)
print("STEP 5 ACTION: Loading full table...")
df = pd.read_sql_query("SELECT * FROM predicting_sales", conn)
conn.close()
print(f"Loaded shape: {df.shape}")

# Action: Convert Date TEXT to real datetime - needed for extracting Year/Month
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Check: Sort by Date ascending - so first rows are 2013, last rows 2015 (time-based split)
df = df.sort_values('Date').reset_index(drop=True)
print(f"\nSTEP 5 CHECK: Sorted by Date")
print(f"Min Date after sort: {df['Date'].min()}, Max Date: {df['Date'].max()}")
print(f"First row Date: {df['Date'].iloc[0]}, Last row Date: {df['Date'].iloc[-1]}")

# Action: Feature Engineering - create new smart columns from Date and holidays
print("\nSTEP 5 ACTION: Creating new features...")

# From Date, extract Year, Month, Day, WeekOfYear, Quarter - these help model see seasonality
df['Year'] = df['Date'].dt.year # Year like 2013, 2014, 2015
df['Month'] = df['Date'].dt.month # Month 1-12, sales higher in Dec
df['Day'] = df['Date'].dt.day # Day of month 1-31
df['WeekOfYear'] = df['Date'].dt.isocalendar().week # Week number 1-52
df['Quarter'] = df['Date'].dt.quarter # Quarter 1-4

# IsWeekend - 1 if Saturday or Sunday (DayOfWeek 6,7), else 0. Weekend sales different
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 6 else 0)

# IsMonthStart / IsMonthEnd - people shop more at start/end of month
df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)

# IsStateHoliday binary - 1 if holiday 'a','b','c', 0 if '0' (no holiday)
df['IsStateHoliday'] = df['StateHoliday'].apply(lambda x: 0 if x == '0' else 1)

# Encode StateHoliday text to numbers for model - '0'=0, 'a'=1, 'b'=2, 'c'=3
state_map = {'0':0, 'a':1, 'b':2, 'c':3}
df['StateHolidayEncoded'] = df['StateHoliday'].map(state_map)

# Check: Show new columns created
print("New features created: Year, Month, Day, WeekOfYear, Quarter, IsWeekend, IsMonthStart, IsMonthEnd, IsStateHoliday, StateHolidayEncoded")
print(df[['Date','Year','Month','DayOfWeek','IsWeekend','StateHoliday','IsStateHoliday']].head(3))

# Action: For honest regression, drop Customers to avoid leakage (future customers unknown)
# Also drop original text StateHoliday and Date (we have encoded versions)
print("\nSTEP 5 ACTION: Dropping Customers (leakage) and keeping only model-ready columns")

# Keep only Open=1 for training? Actually keep all for split, but model will learn Open=0 => Sales=0
# For features, we will use: Store, DayOfWeek, Open, Promo, SchoolHoliday, Year, Month, Day, WeekOfYear, Quarter, IsWeekend, IsStateHoliday, StateHolidayEncoded
feature_cols = ['Store','DayOfWeek','Open','Promo','SchoolHoliday','Year','Month','Day','WeekOfYear','Quarter','IsWeekend','IsMonthStart','IsMonthEnd','IsStateHoliday','StateHolidayEncoded']
target_col = 'Sales'

# Check: Split as per brief - first 700k train, next 200k eval, rest live
print("\nSTEP 5 CHECK: Splitting as per brief (700k train / 200k eval / rest live):")
train_df = df.iloc[:700000].copy() # First 700k rows
eval_df = df.iloc[700000:900000].copy() # Next 200k rows (700k to 900k)
live_df = df.iloc[900000:].copy() # Remaining ~117k rows

print(f"Train shape: {train_df.shape} (first 700k)")
print(f"Eval shape: {eval_df.shape} (next 200k)")
print(f"Live shape: {live_df.shape} (remaining)")

# Check: In train, how many Open=0? Model should predict 0 for those easily
print(f"\nIn Train, Open=0 count: {(train_df['Open']==0).sum()} (these will be predicted as 0)")
print(f"In Eval, Open=0 count: {(eval_df['Open']==0).sum()}")

# Save these DataFrames to reuse in next step - store in pickle or keep in memory
# For Colab, we keep variables train_df, eval_df, live_df in memory for Step 6
print("\nSTEP 5 COMPLETE: Feature engineering done, splits ready")
print(f"Feature columns for model: {feature_cols}")

STEP 5 CHECK: DB exists? True
STEP 5 ACTION: Loading full table...
Loaded shape: (1017209, 9)

STEP 5 CHECK: Sorted by Date
Min Date after sort: 2013-01-01 00:00:00, Max Date: 2015-07-31 00:00:00
First row Date: 2013-01-01 00:00:00, Last row Date: 2015-07-31 00:00:00

STEP 5 ACTION: Creating new features...
New features created: Year, Month, Day, WeekOfYear, Quarter, IsWeekend, IsMonthStart, IsMonthEnd, IsStateHoliday, StateHolidayEncoded
        Date  Year  Month  DayOfWeek  IsWeekend StateHoliday  IsStateHoliday
0 2013-01-01  2013      1          2          0            a               1
1 2013-01-01  2013      1          2          0            a               1
2 2013-01-01  2013      1          2          0            a               1

STEP 5 ACTION: Dropping Customers (leakage) and keeping only model-ready columns

STEP 5 CHECK: Splitting as per brief (700k train / 200k eval / rest live):
Train shape: (700000, 19) (first 700k)
Eval shape: (200000, 19) (next 200k)
Live shape: (11

In [14]:
# ================= STEP 6: Linear Regression - Baseline Model =================
# Trains Linear Regression and evaluates on 200k eval

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import time
import joblib
import os

# Check if train_df and eval_df exist
print("STEP 6 CHECK: Checking train_df/eval_df...")
try:
    print(f"Found Train: {train_df.shape}, Eval: {eval_df.shape}")
except NameError:
    print("Not in memory - reloading...")
    import sqlite3
    db_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/regression.db'
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM predicting_sales", conn)
    conn.close()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week
    df['Quarter'] = df['Date'].dt.quarter
    df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x>=6 else 0)
    df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
    df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)
    df['IsStateHoliday'] = df['StateHoliday'].apply(lambda x: 0 if x=='0' else 1)
    df['StateHolidayEncoded'] = df['StateHoliday'].map({'0':0,'a':1,'b':2,'c':3})
    train_df = df.iloc[:700000].copy()
    eval_df = df.iloc[700000:900000].copy()
    print(f"Reloaded Train: {train_df.shape}, Eval: {eval_df.shape}")

# Define features
feature_cols = ['Store','DayOfWeek','Open','Promo','SchoolHoliday','Year','Month','Day','WeekOfYear','Quarter','IsWeekend','IsMonthStart','IsMonthEnd','IsStateHoliday','StateHolidayEncoded']
target_col = 'Sales'

# Prepare data - only Open=1 for training
print("\nSTEP 6 ACTION: Preparing data (Open=1 only)...")
train_open = train_df[train_df['Open']==1].copy()
X_train = train_open[feature_cols]
y_train = train_open[target_col]
print(f"X_train shape: {X_train.shape}, y_train mean: {y_train.mean():.2f}")

eval_open = eval_df[eval_df['Open']==1].copy()
X_eval_open = eval_open[feature_cols]
print(f"Eval total: {len(eval_df)}, Open=1: {len(eval_open)}, Open=0: {(eval_df['Open']==0).sum()}")

# Train Linear Regression
print("\nSTEP 6 ACTION: Training Linear Regression...")
start = time.time()
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
print(f"Training done in {time.time()-start:.2f} seconds")

# Predict on eval - FIX for dtype warning: init as float 0.0
print("\nSTEP 6 ACTION: Predicting on Eval 200k...")
pred_open = lin_model.predict(X_eval_open)

# Initialize as float to avoid FutureWarning
eval_df['Pred_Lin'] = 0.0
eval_df.loc[eval_df['Open']==1, 'Pred_Lin'] = pred_open.astype(float)

# Metrics
rmse = np.sqrt(mean_squared_error(eval_df['Sales'], eval_df['Pred_Lin']))
mae = mean_absolute_error(eval_df['Sales'], eval_df['Pred_Lin'])
print(f"\nSTEP 6 CHECK: Linear Regression Metrics on 200k Eval:")
print(f"RMSE: {rmse:.2f} dollars")
print(f"MAE: {mae:.2f} dollars")
print(f"Mean Sales in Eval: {eval_df['Sales'].mean():.2f}, Error %: {rmse/eval_df['Sales'].mean()*100:.1f}%")

# Sample actual vs predicted
print("\nSTEP 6 CHECK: Sample Actual vs Predicted (first 5 open stores):")
sample = eval_df[eval_df['Open']==1][['Store','Date','Sales','Pred_Lin']].head(5)
print(sample)

# Save model and metrics for comparison
save_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/model_step6_linear.pkl'
joblib.dump(lin_model, save_path)
with open('/content/drive/MyDrive/INTTRVU/CAPSTONES/results_step6.txt','w') as f:
    f.write(f"Linear Regression RMSE: {rmse}\nMAE: {mae}\n")

print(f"\nSTEP 6 COMPLETE: Model saved to {save_path}")

STEP 6 CHECK: Checking train_df/eval_df...
Found Train: (700000, 19), Eval: (200000, 20)

STEP 6 ACTION: Preparing data (Open=1 only)...
X_train shape: (580965, 15), y_train mean: 6836.55
Eval total: 200000, Open=1: 166474, Open=0: 33526

STEP 6 ACTION: Training Linear Regression...
Training done in 0.53 seconds

STEP 6 ACTION: Predicting on Eval 200k...

STEP 6 CHECK: Linear Regression Metrics on 200k Eval:
RMSE: 2689.71 dollars
MAE: 1757.06 dollars
Mean Sales in Eval: 6030.74, Error %: 44.6%

STEP 6 CHECK: Sample Actual vs Predicted (first 5 open stores):
        Store       Date  Sales     Pred_Lin
700000    956 2014-10-06   5458  8968.370068
700001    955 2014-10-06   6867  8968.281049
700002    954 2014-10-06  11466  8968.192029
700003    953 2014-10-06   8238  8968.103010
700004    951 2014-10-06  10348  8967.924970

STEP 6 COMPLETE: Model saved to /content/drive/MyDrive/INTTRVU/CAPSTONES/model_step6_linear.pkl


In [15]:
# ================= STEP 7: Random Forest Regressor =================
# Trains Random Forest and evaluates on 200k eval

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import time
import joblib

# Check data from Step 5
print("STEP 7 CHECK: Checking train_df/eval_df...")
print(f"Train: {train_df.shape}, Eval: {eval_df.shape}")

# Define features
feature_cols = ['Store','DayOfWeek','Open','Promo','SchoolHoliday','Year','Month','Day','WeekOfYear','Quarter','IsWeekend','IsMonthStart','IsMonthEnd','IsStateHoliday','StateHolidayEncoded']

# Prepare data - Open=1 only for training
print("\nSTEP 7 ACTION: Preparing data (Open=1 only)...")
train_open = train_df[train_df['Open']==1].copy()
X_train = train_open[feature_cols]
y_train = train_open['Sales']
print(f"X_train shape: {X_train.shape}")

eval_open = eval_df[eval_df['Open']==1].copy()
X_eval_open = eval_open[feature_cols]
print(f"Eval total: {len(eval_df)}, Open=1: {len(eval_open)}, Open=0: {(eval_df['Open']==0).sum()}")

# Train Random Forest
print("\nSTEP 7 ACTION: Training Random Forest (50 trees, may take 2-3 min)...")
start = time.time()
rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)
print(f"Training done in {time.time()-start:.1f} seconds")

# Predict - init as float to avoid dtype warning
print("\nSTEP 7 ACTION: Predicting on Eval...")
pred_open = rf_model.predict(X_eval_open)

eval_df['Pred_RF'] = 0.0
eval_df.loc[eval_df['Open']==1, 'Pred_RF'] = pred_open.astype(float)

# Metrics
rmse = np.sqrt(mean_squared_error(eval_df['Sales'], eval_df['Pred_RF']))
mae = mean_absolute_error(eval_df['Sales'], eval_df['Pred_RF'])
print(f"\nSTEP 7 CHECK: Random Forest Metrics on 200k Eval:")
print(f"RMSE: {rmse:.2f} dollars")
print(f"MAE: {mae:.2f} dollars")
print(f"Mean Sales: {eval_df['Sales'].mean():.2f}, Error %: {rmse/eval_df['Sales'].mean()*100:.1f}%")

# Sample actual vs predicted
print("\nSTEP 7 CHECK: Sample Actual vs Predicted:")
sample = eval_df[eval_df['Open']==1][['Store','Date','Sales','Pred_RF']].head(5)
print(sample)

# Feature importance
print("\nSTEP 7 CHECK: Feature importance (higher = more useful):")
imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(imp_df.head(8))

# Save model and metrics
save_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/model_step7_rf.pkl'
joblib.dump(rf_model, save_path)
with open('/content/drive/MyDrive/INTTRVU/CAPSTONES/results_step7.txt','w') as f:
    f.write(f"Random Forest RMSE: {rmse}\nMAE: {mae}\n")

print(f"\nSTEP 7 COMPLETE: Model saved to {save_path}")

STEP 7 CHECK: Checking train_df/eval_df...
Train: (700000, 19), Eval: (200000, 20)

STEP 7 ACTION: Preparing data (Open=1 only)...
X_train shape: (580965, 15)
Eval total: 200000, Open=1: 166474, Open=0: 33526

STEP 7 ACTION: Training Random Forest (50 trees, may take 2-3 min)...
Training done in 80.7 seconds

STEP 7 ACTION: Predicting on Eval...

STEP 7 CHECK: Random Forest Metrics on 200k Eval:
RMSE: 2424.58 dollars
MAE: 1568.94 dollars
Mean Sales: 6030.74, Error %: 40.2%

STEP 7 CHECK: Sample Actual vs Predicted:
        Store       Date  Sales      Pred_RF
700000    956 2014-10-06   5458  8513.822121
700001    955 2014-10-06   6867  8800.498272
700002    954 2014-10-06  11466  9325.122412
700003    953 2014-10-06   8238  9222.128363
700004    951 2014-10-06  10348  9355.927554

STEP 7 CHECK: Feature importance (higher = more useful):
          Feature  Importance
0           Store    0.522180
3           Promo    0.262447
1       DayOfWeek    0.083738
8      WeekOfYear    0.052031
7

In [16]:
# ================= STEP 8: Gradient Boosting Regressor =================
# Trains Gradient Boosting and evaluates on 200k eval

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import time
import joblib

# Check data
print("STEP 8 CHECK: Checking train_df/eval_df...")
print(f"Train: {train_df.shape}, Eval: {eval_df.shape}")

# Define features
feature_cols = ['Store','DayOfWeek','Open','Promo','SchoolHoliday','Year','Month','Day','WeekOfYear','Quarter','IsWeekend','IsMonthStart','IsMonthEnd','IsStateHoliday','StateHolidayEncoded']

# Prepare data
print("\nSTEP 8 ACTION: Preparing data (Open=1 only)...")
train_open = train_df[train_df['Open']==1].copy()
X_train_full = train_open[feature_cols]
y_train_full = train_open['Sales']
print(f"Full X_train: {X_train_full.shape}")

# Sample 150k for speed - Gradient Boosting is slow on full 580k
X_train = X_train_full.sample(n=150000, random_state=42)
y_train = y_train_full.loc[X_train.index]
print(f"Sampled X_train for Gradient Boosting: {X_train.shape}")

eval_open = eval_df[eval_df['Open']==1].copy()
X_eval_open = eval_open[feature_cols]
print(f"Eval total: {len(eval_df)}, Open=1: {len(eval_open)}")

# Train Gradient Boosting
print("\nSTEP 8 ACTION: Training Gradient Boosting (100 trees)...")
start = time.time()
gb_model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)
gb_model.fit(X_train, y_train)
print(f"Training done in {time.time()-start:.1f} seconds")

# Predict
print("\nSTEP 8 ACTION: Predicting on Eval...")
pred_open = gb_model.predict(X_eval_open)

eval_df['Pred_GB'] = 0.0
eval_df.loc[eval_df['Open']==1, 'Pred_GB'] = pred_open.astype(float)

# Metrics
rmse = np.sqrt(mean_squared_error(eval_df['Sales'], eval_df['Pred_GB']))
mae = mean_absolute_error(eval_df['Sales'], eval_df['Pred_GB'])
print(f"\nSTEP 8 CHECK: Gradient Boosting Metrics on 200k Eval:")
print(f"RMSE: {rmse:.2f} dollars")
print(f"MAE: {mae:.2f} dollars")
print(f"Mean Sales: {eval_df['Sales'].mean():.2f}, Error %: {rmse/eval_df['Sales'].mean()*100:.1f}%")

# Sample
print("\nSTEP 8 CHECK: Sample Actual vs Predicted:")
sample = eval_df[eval_df['Open']==1][['Store','Date','Sales','Pred_GB']].head(5)
print(sample)

# Feature importance
print("\nSTEP 8 CHECK: Feature importance:")
imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': gb_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(imp_df.head(8))

# Save
save_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/model_step8_gb.pkl'
joblib.dump(gb_model, save_path)
with open('/content/drive/MyDrive/INTTRVU/CAPSTONES/results_step8.txt','w') as f:
    f.write(f"Gradient Boosting RMSE: {rmse}\nMAE: {mae}\n")

print(f"\nSTEP 8 COMPLETE: Model saved to {save_path}")

STEP 8 CHECK: Checking train_df/eval_df...
Train: (700000, 19), Eval: (200000, 21)

STEP 8 ACTION: Preparing data (Open=1 only)...
Full X_train: (580965, 15)
Sampled X_train for Gradient Boosting: (150000, 15)
Eval total: 200000, Open=1: 166474

STEP 8 ACTION: Training Gradient Boosting (100 trees)...
Training done in 35.8 seconds

STEP 8 ACTION: Predicting on Eval...

STEP 8 CHECK: Gradient Boosting Metrics on 200k Eval:
RMSE: 2093.97 dollars
MAE: 1406.64 dollars
Mean Sales: 6030.74, Error %: 34.7%

STEP 8 CHECK: Sample Actual vs Predicted:
        Store       Date  Sales      Pred_GB
700000    956 2014-10-06   5458  9015.231588
700001    955 2014-10-06   6867  9015.231588
700002    954 2014-10-06  11466  9128.501813
700003    953 2014-10-06   8238  9128.501813
700004    951 2014-10-06  10348  9128.501813

STEP 8 CHECK: Feature importance:
          Feature  Importance
0           Store    0.538039
3           Promo    0.259384
1       DayOfWeek    0.080035
8      WeekOfYear    0.0440

In [17]:
# ================= STEP 9: XGBoost Regressor =================
# Trains XGBoost and evaluates on 200k eval

import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import time
import joblib

# Check data
print("STEP 9 CHECK: Checking train_df/eval_df...")
print(f"Train: {train_df.shape}, Eval: {eval_df.shape}")

# Define features
feature_cols = ['Store','DayOfWeek','Open','Promo','SchoolHoliday','Year','Month','Day','WeekOfYear','Quarter','IsWeekend','IsMonthStart','IsMonthEnd','IsStateHoliday','StateHolidayEncoded']

# Prepare data - Open=1 only
print("\nSTEP 9 ACTION: Preparing data (Open=1 only)...")
train_open = train_df[train_df['Open']==1].copy()
X_train = train_open[feature_cols]
y_train = train_open['Sales']
print(f"X_train: {X_train.shape}")

eval_open = eval_df[eval_df['Open']==1].copy()
X_eval_open = eval_open[feature_cols]
print(f"Eval total: {len(eval_df)}, Open=1: {len(eval_open)}")

# Train XGBoost
print("\nSTEP 9 ACTION: Training XGBoost (200 trees)...")
start = time.time()
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    n_jobs=-1,
    random_state=42
)
xgb_model.fit(X_train, y_train)
print(f"Training done in {time.time()-start:.1f} seconds")

# Predict
print("\nSTEP 9 ACTION: Predicting on Eval...")
pred_open = xgb_model.predict(X_eval_open)

eval_df['Pred_XGB'] = 0.0
eval_df.loc[eval_df['Open']==1, 'Pred_XGB'] = pred_open.astype(float)

# Metrics
rmse = np.sqrt(mean_squared_error(eval_df['Sales'], eval_df['Pred_XGB']))
mae = mean_absolute_error(eval_df['Sales'], eval_df['Pred_XGB'])
print(f"\nSTEP 9 CHECK: XGBoost Metrics on 200k Eval:")
print(f"RMSE: {rmse:.2f} dollars")
print(f"MAE: {mae:.2f} dollars")
print(f"Mean Sales: {eval_df['Sales'].mean():.2f}, Error %: {rmse/eval_df['Sales'].mean()*100:.1f}%")

# Sample
print("\nSTEP 9 CHECK: Sample Actual vs Predicted:")
sample = eval_df[eval_df['Open']==1][['Store','Date','Sales','Pred_XGB']].head(5)
print(sample)

# Feature importance
print("\nSTEP 9 CHECK: Feature importance:")
imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(imp_df.head(8))

# Save
save_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/model_step9_xgb.pkl'
joblib.dump(xgb_model, save_path)
with open('/content/drive/MyDrive/INTTRVU/CAPSTONES/results_step9.txt','w') as f:
    f.write(f"XGBoost RMSE: {rmse}\nMAE: {mae}\n")

print(f"\nSTEP 9 COMPLETE: Model saved to {save_path}")

STEP 9 CHECK: Checking train_df/eval_df...
Train: (700000, 19), Eval: (200000, 22)

STEP 9 ACTION: Preparing data (Open=1 only)...
X_train: (580965, 15)
Eval total: 200000, Open=1: 166474

STEP 9 ACTION: Training XGBoost (200 trees)...
Training done in 11.6 seconds

STEP 9 ACTION: Predicting on Eval...

STEP 9 CHECK: XGBoost Metrics on 200k Eval:
RMSE: 2520.00 dollars
MAE: 1643.50 dollars
Mean Sales: 6030.74, Error %: 41.8%

STEP 9 CHECK: Sample Actual vs Predicted:
        Store       Date  Sales     Pred_XGB
700000    956 2014-10-06   5458  8506.574219
700001    955 2014-10-06   6867  8506.574219
700002    954 2014-10-06  11466  8506.574219
700003    953 2014-10-06   8238  9027.763672
700004    951 2014-10-06  10348  9027.763672

STEP 9 CHECK: Feature importance:
           Feature  Importance
3            Promo    0.612210
0            Store    0.105773
1        DayOfWeek    0.070250
12      IsMonthEnd    0.044719
8       WeekOfYear    0.036041
7              Day    0.034128
6      

In [18]:
# ================= STEP 10: Final Comparison - Pick Best Model =================
# Compares all 4 models from Steps 6-9 and saves best model

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import joblib
import os

# Check eval_df has all 4 predictions
print("STEP 10 CHECK: Checking Eval predictions from Steps 6-9...")
print(f"Eval shape: {eval_df.shape}")
print(f"Columns: {list(eval_df.columns)}")
has_lin = 'Pred_Lin' in eval_df.columns
has_rf = 'Pred_RF' in eval_df.columns
has_gb = 'Pred_GB' in eval_df.columns
has_xgb = 'Pred_XGB' in eval_df.columns
print(f"Has Pred_Lin: {has_lin}, Pred_RF: {has_rf}, Pred_GB: {has_gb}, Pred_XGB: {has_xgb}")

# If any missing, load from saved results files
if not (has_lin and has_rf and has_gb and has_xgb):
    print("Some predictions missing - will compare from saved txt files")

# Calculate RMSE and MAE for each model on full Eval (including Open=0 as 0)
print("\nSTEP 10 ACTION: Calculating metrics for all 4 models...")

results = []

# Linear
if has_lin:
    rmse_lin = np.sqrt(mean_squared_error(eval_df['Sales'], eval_df['Pred_Lin']))
    mae_lin = mean_absolute_error(eval_df['Sales'], eval_df['Pred_Lin'])
    results.append(['Linear Regression', rmse_lin, mae_lin])
    print(f"Linear Regression - RMSE: {rmse_lin:.2f}, MAE: {mae_lin:.2f}")

# Random Forest
if has_rf:
    rmse_rf = np.sqrt(mean_squared_error(eval_df['Sales'], eval_df['Pred_RF']))
    mae_rf = mean_absolute_error(eval_df['Sales'], eval_df['Pred_RF'])
    results.append(['Random Forest', rmse_rf, mae_rf])
    print(f"Random Forest - RMSE: {rmse_rf:.2f}, MAE: {mae_rf:.2f}")

# Gradient Boosting
if has_gb:
    rmse_gb = np.sqrt(mean_squared_error(eval_df['Sales'], eval_df['Pred_GB']))
    mae_gb = mean_absolute_error(eval_df['Sales'], eval_df['Pred_GB'])
    results.append(['Gradient Boosting', rmse_gb, mae_gb])
    print(f"Gradient Boosting - RMSE: {rmse_gb:.2f}, MAE: {mae_gb:.2f}")

# XGBoost
if has_xgb:
    rmse_xgb = np.sqrt(mean_squared_error(eval_df['Sales'], eval_df['Pred_XGB']))
    mae_xgb = mean_absolute_error(eval_df['Sales'], eval_df['Pred_XGB'])
    results.append(['XGBoost', rmse_xgb, mae_xgb])
    print(f"XGBoost - RMSE: {rmse_xgb:.2f}, MAE: {mae_xgb:.2f}")

# Comparison table sorted by RMSE
comparison_df = pd.DataFrame(results, columns=['Model','RMSE','MAE']).sort_values('RMSE')
print("\nSTEP 10 CHECK: Final Comparison Table (sorted by RMSE, lower is better):")
print(comparison_df.to_string(index=False))

# Winner
best_model_name = comparison_df.iloc[0]['Model']
best_rmse = comparison_df.iloc[0]['RMSE']
print(f"\nSTEP 10 CHECK: Winner is {best_model_name} with RMSE {best_rmse:.2f}")

# Save comparison for report
comparison_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/model_comparison.csv'
comparison_df.to_csv(comparison_path, index=False)
print(f"Comparison saved to {comparison_path}")

# Load best model file and copy as best_sales_model.pkl
model_map = {
    'Linear Regression': '/content/drive/MyDrive/INTTRVU/CAPSTONES/model_step6_linear.pkl',
    'Random Forest': '/content/drive/MyDrive/INTTRVU/CAPSTONES/model_step7_rf.pkl',
    'Gradient Boosting': '/content/drive/MyDrive/INTTRVU/CAPSTONES/model_step8_gb.pkl',
    'XGBoost': '/content/drive/MyDrive/INTTRVU/CAPSTONES/model_step9_xgb.pkl'
}

best_path = model_map.get(best_model_name)
if best_path and os.path.exists(best_path):
    best_model = joblib.load(best_path)
    final_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/best_sales_model.pkl'
    joblib.dump(best_model, final_path)
    print(f"Best model ({best_model_name}) copied to {final_path}")
else:
    print(f"Best model file not found at {best_path}")

# Improvement over baseline
baseline_rmse = comparison_df[comparison_df['Model']=='Linear Regression']['RMSE'].values[0] if 'Linear Regression' in comparison_df['Model'].values else None
if baseline_rmse:
    improvement = (baseline_rmse - best_rmse) / baseline_rmse * 100
    print(f"Improvement over Linear baseline: {improvement:.1f}%")

print("\nSTEP 10 COMPLETE: Comparison done, winner selected and saved")

STEP 10 CHECK: Checking Eval predictions from Steps 6-9...
Eval shape: (200000, 23)
Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'Quarter', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd', 'IsStateHoliday', 'StateHolidayEncoded', 'Pred_Lin', 'Pred_RF', 'Pred_GB', 'Pred_XGB']
Has Pred_Lin: True, Pred_RF: True, Pred_GB: True, Pred_XGB: True

STEP 10 ACTION: Calculating metrics for all 4 models...
Linear Regression - RMSE: 2689.71, MAE: 1757.06
Random Forest - RMSE: 2424.58, MAE: 1568.94
Gradient Boosting - RMSE: 2093.97, MAE: 1406.64
XGBoost - RMSE: 2520.00, MAE: 1643.50

STEP 10 CHECK: Final Comparison Table (sorted by RMSE, lower is better):
            Model        RMSE         MAE
Gradient Boosting 2093.967633 1406.637493
    Random Forest 2424.575280 1568.940928
          XGBoost 2520.000285 1643.501438
Linear Regression 2689.710534 1757.057106

STEP 10 CHECK: Winner is Gradient Boosti

In [19]:
# ================= STEP 11: Live Data Prediction - Final Assessment =================
# Predicts on remaining 117209 live rows using best model from Step 10

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import joblib
import os

# Check train_df, eval_df, live_df
print("STEP 11 CHECK: Checking data...")
try:
    print(f"Train: {train_df.shape}, Eval: {eval_df.shape}")
    # Try live_df from Step 5
    print(f"Live: {live_df.shape}")
except NameError:
    print("live_df not in memory - recreating from DB...")
    import sqlite3
    db_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/regression.db'
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM predicting_sales", conn)
    conn.close()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week
    df['Quarter'] = df['Date'].dt.quarter
    df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x>=6 else 0)
    df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
    df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)
    df['IsStateHoliday'] = df['StateHoliday'].apply(lambda x: 0 if x=='0' else 1)
    df['StateHolidayEncoded'] = df['StateHoliday'].map({'0':0,'a':1,'b':2,'c':3})
    train_df = df.iloc[:700000].copy()
    eval_df = df.iloc[700000:900000].copy()
    live_df = df.iloc[900000:].copy()
    print(f"Recreated Train: {train_df.shape}, Eval: {eval_df.shape}, Live: {live_df.shape}")

# Define features
feature_cols = ['Store','DayOfWeek','Open','Promo','SchoolHoliday','Year','Month','Day','WeekOfYear','Quarter','IsWeekend','IsMonthStart','IsMonthEnd','IsStateHoliday','StateHolidayEncoded']

# Check live
print(f"\nSTEP 11 CHECK: Live data details:")
print(f"Live shape: {live_df.shape}")
print(f"Live Date range: {live_df['Date'].min()} to {live_df['Date'].max()}")
print(f"Live Open=0 count: {(live_df['Open']==0).sum()} (will be predicted as 0)")

# Load best model from Step 10
model_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/best_sales_model.pkl'
print(f"\nSTEP 11 ACTION: Loading best model from {model_path}...")
best_model = joblib.load(model_path)
print(f"Best model loaded: {type(best_model).__name__}")

# Prepare live data
print("\nSTEP 11 ACTION: Preparing live data...")
live_open = live_df[live_df['Open']==1].copy()
X_live_open = live_open[feature_cols]
print(f"Live Open=1 rows to predict: {len(X_live_open)}")

# Predict
print("\nSTEP 11 ACTION: Predicting Sales on live...")
pred_live_open = best_model.predict(X_live_open)

# Create final prediction column - float to avoid warning
live_df['PredictedSales'] = 0.0
live_df.loc[live_df['Open']==1, 'PredictedSales'] = pred_live_open.astype(float)

# Metrics on live - we have true Sales for assessment
rmse_live = np.sqrt(mean_squared_error(live_df['Sales'], live_df['PredictedSales']))
mae_live = mean_absolute_error(live_df['Sales'], live_df['PredictedSales'])
print(f"\nSTEP 11 CHECK: Live Data Metrics (final assessment):")
print(f"RMSE: {rmse_live:.2f} dollars")
print(f"MAE: {mae_live:.2f} dollars")
print(f"Mean Sales in Live: {live_df['Sales'].mean():.2f}, Error %: {rmse_live/live_df['Sales'].mean()*100:.1f}%")

# Sample actual vs predicted on live
print("\nSTEP 11 CHECK: Sample Live Actual vs Predicted:")
sample_live = live_df[live_df['Open']==1][['Store','Date','Sales','PredictedSales']].head(8)
print(sample_live)

# Save live predictions for submission
save_live_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/live_predictions.csv'
live_df[['Date','Store','Sales','PredictedSales']].to_csv(save_live_path, index=False)
print(f"\nSTEP 11 CHECK: Live predictions saved to {save_live_path}")

# Save final report summary
report_path = '/content/drive/MyDrive/INTTRVU/CAPSTONES/final_report.txt'
with open(report_path,'w') as f:
    f.write("Sales Prediction Project - Final Report\n")
    f.write(f"Train: 700k, Eval: 200k, Live: {len(live_df)}\n")
    f.write(f"Features: {feature_cols}\n")
    f.write(f"Linear RMSE: 2689.71\n")
    f.write(f"Random Forest RMSE: 2424.58\n")
    f.write(f"Gradient Boosting RMSE: 2093.97 (Winner)\n")
    f.write(f"XGBoost RMSE: 2520.00\n")
    f.write(f"Winner Improvement over Linear: 22.1%\n")
    f.write(f"Live RMSE with Winner: {rmse_live:.2f}\n")
    f.write(f"Live MAE with Winner: {mae_live:.2f}\n")

print(f"Final report saved to {report_path}")
print("\nSTEP 11 COMPLETE: Live prediction done")

STEP 11 CHECK: Checking data...
Train: (700000, 19), Eval: (200000, 23)
Live: (117209, 19)

STEP 11 CHECK: Live data details:
Live shape: (117209, 19)
Live Date range: 2015-04-17 00:00:00 to 2015-07-31 00:00:00
Live Open=0 count: 20256 (will be predicted as 0)

STEP 11 ACTION: Loading best model from /content/drive/MyDrive/INTTRVU/CAPSTONES/best_sales_model.pkl...
Best model loaded: GradientBoostingRegressor

STEP 11 ACTION: Preparing live data...
Live Open=1 rows to predict: 96953

STEP 11 ACTION: Predicting Sales on live...

STEP 11 CHECK: Live Data Metrics (final assessment):
RMSE: 1979.48 dollars
MAE: 1323.06 dollars
Mean Sales in Live: 5931.62, Error %: 33.4%

STEP 11 CHECK: Sample Live Actual vs Predicted:
        Store       Date  Sales  PredictedSales
900000    795 2015-04-17   5307     7603.403649
900001    794 2015-04-17   3623     7603.403649
900002    793 2015-04-17   6030     7603.403649
900003    792 2015-04-17   8250     7763.616292
900004    791 2015-04-17   8633     77